In [ ]:
# [Cell 1] 구글 드라이브 마운트 및 작업 디렉토리 설정
import os
from google.colab import drive

# 1. 드라이브 마운트
drive.mount('/content/drive')

# 2. 프로젝트 전용 폴더 생성 (내 드라이브 -> Glucose_AI_Project)
PROJECT_DIR = '/content/drive/MyDrive/Glucose_AI_Project'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs')

for path in [DATA_DIR, MODEL_DIR, OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)

print(f"구글 드라이브 마운트 완료!")
print(f"모델 저장 경로: {MODEL_DIR}")
print(f"TFLite 파일 내보내기 경로: {OUTPUT_DIR}")

In [ ]:
# [Cell 2] ONNX 및 TFLite 변환 라이브러리 설치
!pip install -q onnx onnx2tf tensorflow scikit-learn pandas numpy matplotlib
print("필수 라이브러리 설치 completed!")

#Cell 3 Change Log
정확한 사전학습데이터 없어서 미리 생성함

In [ ]:
# [Cell 3] PyTorch Dataset 및 가상 데이터 생성기 정의
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class GlucoseDataset(Dataset):
    def __init__(self, num_samples=1000, sequence_length=100, is_optical_only=False):
        """
        - num_samples: 샘플 개수
        - sequence_length: 10초 시계열 샘플링 개수 (예: 10Hz = 100)
        - is_optical_only: 사전 학습용(NIR/PPG 3채널) 또는 전체 학습용(7채널) 선택
        """
        self.is_optical_only = is_optical_only
        num_channels = 3 if is_optical_only else 7

        # 1. 시계열 파형 데이터 생성 (Batch, Channels, Length)
        self.wave_data = np.random.randn(num_samples, num_channels, sequence_length).astype(np.float32)

        # 2. Cole-Cole 물리 파라미터 & 온도/습도 스칼라 데이터 (Batch, 6)
        self.phys_data = np.random.randn(num_samples, 6).astype(np.float32)

        # 3. 정답 혈당 수치 mg/dL (70 ~ 200 mg/dL 범주의 정규 분포)
        self.labels = np.random.uniform(70.0, 200.0, size=(num_samples, 1)).astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if self.is_optical_only:
            return torch.tensor(self.wave_data[idx]), torch.tensor(self.labels[idx])
        return torch.tensor(self.wave_data[idx]), torch.tensor(self.phys_data[idx]), torch.tensor(self.labels[idx])

# 데이터로더 생성
train_loader_full = DataLoader(GlucoseDataset(num_samples=800, is_optical_only=False), batch_size=32, shuffle=True)
val_loader_full = DataLoader(GlucoseDataset(num_samples=200, is_optical_only=False), batch_size=32, shuffle=False)

print("데이터셋 준비 완료 (7채널 샘플: 800개 학습 / 200개 검증)")

In [ ]:
# [Cell 4] PyTorch 1D-ResNet + TCN + MLP 융합 모델
import torch.nn as nn
import torch.nn.functional as F

# 1. 시계열 1D Residual Block
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        res = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += res
        return F.relu(out)

# 2. 전체 멀티모달 혈당 예측 모델
class MultimodalGlucoseModel(nn.Module):
    def __init__(self, in_wave_channels=7, phys_dim=6):
        super(MultimodalGlucoseModel, self).__init__()

        # [파형 브랜치]: 1D-ResNet
        self.wave_branch = nn.Sequential(
            nn.Conv1d(in_wave_channels, 16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            ResidualBlock1D(16, 32, stride=2),
            ResidualBlock1D(32, 64, stride=2),
            nn.AdaptiveAvgPool1d(1) # [Batch, 64, 1]
        )

        # [물리 파라미터 브랜치]: MLP
        self.phys_branch = nn.Sequential(
            nn.Linear(phys_dim, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU()
        )

        # [최종 결합 레이어 (Fusion Head)]: 64 + 16 = 80차원 -> 1차원 혈당 수치
        self.fusion_head = nn.Sequential(
            nn.BatchNorm1d(64 + 16),
            nn.Linear(80, 32),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.SiLU(),
            nn.Linear(16, 1) # 최종 출력: 혈당 수치 (mg/dL)
        )

    def forward(self, wave, phys):
        # 파형 특징 추출
        f_wave = self.wave_branch(wave).squeeze(-1) # [Batch, 64]
        # 물리 파라미터 특징 추출
        f_phys = self.phys_branch(phys)             # [Batch, 16]

        # 결합 레이어 (Concatenation)
        combined = torch.cat((f_wave, f_phys), dim=1) # [Batch, 80]

        # 최종 혈당 예측값 반환
        return self.fusion_head(combined)

print("AI 모델 네트워크 구조 정의 완료!")

In [ ]:
# [Cell 5] PyTorch 모델 학습 및 구글 드라이브 저장
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"학습 실행 장치: {device}")

# 모델, 손실함수, 옵티마이저 초기화
model = MultimodalGlucoseModel(in_wave_channels=7, phys_dim=6).to(device)
criterion = nn.SmoothL1Loss() # 노이즈에 강인한 Huber Loss
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

best_val_loss = float('inf')
best_model_path = os.path.join(MODEL_DIR, 'best_glucose_model.pt')

# 학습 루프 (Epochs: 30)
EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for wave, phys, label in train_loader_full:
        wave, phys, label = wave.to(device), phys.to(device), label.to(device)

        optimizer.zero_grad()
        output = model(wave, phys)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * wave.size(0)

    train_loss /= len(train_loader_full.dataset)

    # 검증(Validation)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for wave, phys, label in val_loader_full:
            wave, phys, label = wave.to(device), phys.to(device), label.to(device)
            output = model(wave, phys)
            loss = criterion(output, label)
            val_loss += loss.item() * wave.size(0)

    val_loss /= len(val_loader_full.dataset)

    # 최고 성능 모델 구글 드라이브에 저장
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_status = "(최고 모델 저장 완료)"
    else:
        saved_status = ""

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{EPOCHS}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} {saved_status}")

print(f"\n학습 완료! 저장된 모델 위치: {best_model_path}")

In [ ]:
# [Cell 6] PyTorch -> ONNX -> TFLite 파이프라인 변환
import torch.onnx
import tensorflow as tf

# 1. 학습된 최고 성능 가중치 로드
model = MultimodalGlucoseModel(in_wave_channels=7, phys_dim=6)
model.load_state_dict(torch.load(best_model_path, map_location='cpu'))
model.eval()

# 2. Dummy Input 생성 (ONNX 변환용 파라미터 형상 지정)
dummy_wave = torch.randn(1, 7, 100) # [Batch=1, 7채널, 10초 시계열 100개]
dummy_phys = torch.randn(1, 6)      # [Batch=1, Cole-Cole 등 6개 파라미터]

onnx_path = os.path.join(OUTPUT_DIR, 'glucose_model.onnx')
tflite_path = os.path.join(OUTPUT_DIR, 'glucose_model.tflite')

# 3. PyTorch -> ONNX 변환
torch.onnx.export(
    model,
    (dummy_wave, dummy_phys),
    onnx_path,
    input_names=['input_wave', 'input_phys'],
    output_names=['output_glucose'],
    opset_version=13
)
print(f" ONNX 변환 성공: {onnx_path}")

# 4. ONNX -> TFLite 변환 실행
!onnx2tf -i {onnx_path} -o {OUTPUT_DIR}/tflite_temp -oiqt

# 5. 생성된 tflite 파일을 최종 경로로 이동
import shutil
generated_tflite = os.path.join(OUTPUT_DIR, 'tflite_temp', 'glucose_model_float32.tflite')
if not os.path.exists(generated_tflite):
    # 정수 양자화 파일 이름 대응
    generated_tflite = [os.path.join(OUTPUT_DIR, 'tflite_temp', f) for f in os.listdir(os.path.join(OUTPUT_DIR, 'tflite_temp')) if f.endswith('.tflite')][0]

shutil.copy(generated_tflite, tflite_path)

print(f"\n [최종 성공] TFLite 파일 내보내기 완료!")
print(f" 구글 드라이브 파일 경로: {tflite_path}")
print(f" 파일 크기: {os.path.getsize(tflite_path) / 1024:.2f} KB")

상단 메뉴 변경: Colab 상단 메뉴에서 런타임 $\to$ 런타임 유형 변경 $\to$ T4 GPU를 선택해 연산 가속을 켭니다.순차적 실행: Cell 1부터 Cell 6까지 순서대로 실행 버튼(▶)을 누릅니다.결과 확인: 실행이 모두 완료되면 내 구글 드라이브의 Glucose_AI_Project/outputs/ 폴더 안에 glucose_model.tflite 파일이 생성되어 있습니다.실제 데이터 적용 시 (Cell 3 수정):Kaggle 데이터셋이나 센서 수집 CSV 데이터를 구글 드라이브 Glucose_AI_Project/data/ 폴더에 업로드합니다.Cell 3의 np.random.randn 부분을 pandas.read_csv() 코드로 변경하여 실제 수집한 데이터 arrays로 교체하신 후 Cell 3부터 다시 돌리시면 됩니다.앱 적용: 내 드라이브에서 glucose_model.tflite 파일을 컴퓨터로 다운로드하여 Flutter 앱의 assets/ 폴더에 옮겨 넣으시면 온디바이스 개발 준비가 완료됩니다.